# **LIBRARIES**

In [1]:
import pandas as pd
import numpy as np
import re
import string
import nltk
from nltk.corpus import stopwords
from sklearn.utils import resample
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.metrics import confusion_matrix, accuracy_score

from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier

ModuleNotFoundError: No module named 'imblearn'

# **LOADING DATA**

In [ ]:
df = pd.read_csv('Roman Urdu Toxic Dataset - Sheet1.csv')
df

,Text,toxic
0,ye moulana diesal sab se bara khabees ha,1
1,local molana ha ye,0
2,paisay ka pujari sala,0
3,Kafi sare loopholes hei scripts mei,0
4,Adeel acha husband nahi hay,0
...,...,...
11364,Ye kafar he polis waly,1
11365,Bagerat police,1
11366,Allah aisy police walo ko tabbah kray gandy gh...,0
11367,Zalim.police waly,0


# **PRE-PROCESSING**

In [ ]:
print("Shape:",df.shape)

# Remove missing values
df.dropna(subset=['Text', 'toxic'], inplace=True)

print("Shape:", df.shape)

Shape: (11369, 2)
Shape: (11369, 2)


In [ ]:
nltk.download('stopwords')

# stopwords
roman_urdu_stopwords = [
    'kya', 'kon', 'kahan', 'kyun', 'nahi', 'haan', 'hai', 'ko', 'ka', 'ke', 'se',
    'mein', 'tum', 'ye', 'wo', 'tha', 'thi', 'hain', 'ho', 'raha', 'rha', 'rhi',
    'aur', 'magar', 'lekin', 'jab', 'ab', 'to', 'bhi', 'mat', 'kar', 'kr', 'kaun',
    'tu', 'teri', 'mera', 'meri', 'us', 'un', 'kis', 'jo', 'ja', 'liya', 'diya',
    'karna', 'hona', 'chal', 'gaya', 'gayee', 'sath', 'aise', 'aisei', 'aisehi',
    'kuch', 'bhot', 'zyada', 'kam', 'kitna', 'ky', 'hi'
]
stop_words = set(stopwords.words('english') + roman_urdu_stopwords)

# Preprocess Function
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"@\w+|#\w+", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DeLL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# **FEATURE EXTRACTION**

In [ ]:
# Balance dataset
count_class_0, count_class_1 = df['toxic'].value_counts()
df_majority = df[df['toxic'] == 0]
df_minority = df[df['toxic'] == 1]

if count_class_0 > count_class_1:
    df_minority_upsampled = resample(df_minority,
                                     replace=True,
                                     n_samples=count_class_0,
                                     random_state=42)
    df = pd.concat([df_majority, df_minority_upsampled])
    print("Data balanced using upsampling.")

Data balanced using upsampling.


In [ ]:
# using N-Grams
vectorizer = TfidfVectorizer(analyzer='char',
    ngram_range=(1, 5),
    max_features=30000,
    min_df=1,
    max_df=0.9,
    sublinear_tf=True)
X = vectorizer.fit_transform(df['Text'])
y = df['toxic']

# **DATA SPLITTING**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3) # 30% testing data

***CLASS BALANCING***

In [ ]:
# SMOTE to balance classes
smote = SMOTE(sampling_strategy='auto', random_state=42, k_neighbors=5)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(X_train_resampled.shape[0], y_train_resampled.shape[0])

10480 10480


# **TRAINING & EVALUATION**

In [ ]:
# Model training and evaluation
def train_and_evaluate(model, name):
    model.fit(X_train_resampled, y_train_resampled) # Training (70%)
    y_pred = model.predict(X_test) # Testing (30%)
    acc = accuracy_score(y_test, y_pred)

    print(f"\n{name} Accuracy: {acc*100:.2f}%")
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
# models
models = {
    'XGBoost': XGBClassifier(learning_rate=0.1, n_estimators=300, max_depth=5, subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', use_label_encoder=False, random_state=42),
    'Multinomial Naive Bayes': MultinomialNB(alpha=0.1),
    'logistic regg': LogisticRegression(class_weight='balanced', max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=30, random_state=42),
    'SVM (RBF Kernel)': SVC(kernel='rbf', gamma=0.1, C=10)
}

for name, model in models.items():
    train_and_evaluate(model, name)

c:\Users\DeLL\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [19:12:53] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



XGBoost Accuracy: 86.63%
Confusion Matrix:
 [[ 948  172]
 [ 284 2007]]

Multinomial Naive Bayes Accuracy: 81.53%
Confusion Matrix:
 [[ 896  224]
 [ 406 1885]]

logistic regg Accuracy: 82.20%
Confusion Matrix:
 [[ 986  134]
 [ 473 1818]]

Random Forest Accuracy: 88.36%
Confusion Matrix:
 [[1009  111]
 [ 286 2005]]

SVM (RBF Kernel) Accuracy: 85.99%
Confusion Matrix:
 [[1026   94]
 [ 384 1907]]


# **TESTING**

In [ ]:
t1 = "khabees ho aap to"
t2 = "ye moulana diesal sab se bara khabees ha"
t3 = "ghatiya makeup ha bohat"
t4 = "Biryani banay ga ye haram Khor"
t5 = "Lanat aesy munafik pr"

n1 = "aap sy milkr acha lga"
n2 = "Allah aapko khush rakhy"
n3 = "bohot hi acha kam kia ha inhu ny"
n4 = "hamy apny aap pr yakeen ha"
n5 = "kya masla hua ha yaha?"

In [ ]:
for name, model in models.items():
    if name == 'Random Forest': # chosen model
								best_model = model
								break

# use input here
comment = t5

# input processing
cleaned_comment = preprocess_text(comment)
comment_vector = vectorizer.transform([cleaned_comment])

# Predict toxic/not toxic
prediction = model.predict(comment_vector)
if prediction[0] == 1:
    print("The comment is toxic.")
else:
    print("The comment is not toxic.")

The comment is toxic.
